In [36]:
from mhctools import NetMHC, NetMHC3,  NetMHCIIpan
from mhcnames import normalize_allele_name## ONLY FOR LINUX

In [57]:
per_allele_fasta = {}
per_decoy_key_per_allele_decoy_fasta = {}
per_decoy_key_decoy_fasta = {}

for decoy_key in DECOY_PREFIXES:
    filename = DECOY_PATH_BASE + "_" + decoy_key + "_" + DECOY_SUFFIX + DECOY_END
    decoy_fasta = readFasta(filename)
    per_decoy_key_decoy_fasta[decoy_key] = decoy_fasta
    
for decoy_key in DECOY_PREFIXES:
    per_decoy_key_per_allele_decoy_fasta[decoy_key] = dict()
    for allele_name in ALLELES:
        filename = binders_filename_base + "_" + allele_name + ".fasta"
        fasta = readFasta(filename)
        per_allele_fasta[allele_name] = fasta
        
        multiplier = int(decoy_key[1:])
        
        per_decoy_key_per_allele_decoy_fasta[decoy_key][allele_name] = dict(list(per_decoy_key_decoy_fasta[decoy_key].items()))


print([(len(per_allele_fasta[allele_name]), len(per_decoy_key_per_allele_decoy_fasta[decoy_key][allele_name])) for allele_name in ALLELES for decoy_key in DECOY_PREFIXES])

[(1438, 1680), (468, 1680), (381, 1680)]


In [58]:
def splitByLengthAndFilter(fasta_dict, min_length = 9):
    unique_lengths = list(set(map(len, list(fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts



In [59]:
per_allele_per_length_dict = dict() 
per_decoy_key_per_allele_per_length_decoy_dict = dict()

for decoy_key in DECOY_PREFIXES:
    per_decoy_key_per_allele_per_length_decoy_dict[decoy_key] = dict()
    for allele_name in ALLELES:
        fasta_dict = per_allele_fasta[allele_name]
        per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        print(f"allele fasta splited into {len(per_length_dict_of_dicts)} dicts")
        per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts


        ## make same for decoy dicts
        fasta_dict = per_decoy_key_per_allele_decoy_fasta[decoy_key][allele_name]
        per_length_decoy_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        print(f"decoy fasta splited into {len(per_length_decoy_dict_of_dicts)} dicts")
        per_decoy_key_per_allele_per_length_decoy_dict[decoy_key][allele_name] = per_length_decoy_dict_of_dicts


# fasta_dict = decoy_fasta
# per_length_decoy_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
# print(f"splited into {len(per_length_decoy_dict_of_dicts)} dicts")




allele fasta splited into 23 dicts
decoy fasta splited into 13 dicts
allele fasta splited into 22 dicts
decoy fasta splited into 13 dicts
allele fasta splited into 17 dicts
decoy fasta splited into 13 dicts


In [60]:
per_allele_per_length_dict[allele_name].keys()

dict_keys([9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25])

In [61]:
ALLELES = list(per_allele_per_length_dict.keys())
ALLELES

['HLA_DRB1_01_01', 'HLA_DRB1_04_01', 'HLA_DRB1_04_05']

In [62]:
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DR':
        target_allele_name = 'HLA_DRB1_01_01'
    elif allele_name == 'HLA_DQ':
        target_allele_name = 'HLA-DQA10101-DQB10201'
    elif allele_name == 'HLA_DQA1_02_01_DQB1_02_02':
        target_allele_name = 'HLA-DQA10201-DQB10202'
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA_DRB1_01_01': 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01': 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05': 'HLA_DRB1_04_05'}

In [63]:
%%time
def predictSequences(dict_of_dicts, allele_name):
    alleles = [normalize_allele_name(TARGET_ALLELES[allele_name])]
    predictor = NetMHCIIpan(
            alleles=alleles)
    print(f"Now we are going to predict binders for  dataset {allele_name} using alleles parameter {alleles}")
    binding_predictions = [predictor.predict_subsequences(fasta_dict, 
                                                          peptide_lengths=[length]) # peek any element and take length
                           for length, fasta_dict in dict_of_dicts.items()]
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions



CPU times: user 17 µs, sys: 3 µs, total: 20 µs
Wall time: 25 µs


In [64]:
per_allele_binder_predictions = dict()
per_decoy_key_per_allele_non_binder_predictions = dict()
for decoy_key in DECOY_PREFIXES:
    per_decoy_key_per_allele_non_binder_predictions[decoy_key] = dict()
    for allele_name, per_length_dict in per_allele_per_length_dict.items():
        predictions = predictSequences(per_length_dict, allele_name)
        per_allele_binder_predictions[allele_name] = predictions
        print("do same for decoy")
        predictions = predictSequences(per_decoy_key_per_allele_per_length_decoy_dict[decoy_key][allele_name], allele_name)
        per_decoy_key_per_allele_non_binder_predictions[decoy_key][allele_name] = predictions
    
    
    


Now we are going to predict binders for  dataset HLA_DRB1_01_01 using alleles parameter ['HLA-DRA1*01:01-DRB1*01:01']
total processed peptides after filtering -  1433
do same for decoy
Now we are going to predict binders for  dataset HLA_DRB1_01_01 using alleles parameter ['HLA-DRA1*01:01-DRB1*01:01']
total processed peptides after filtering -  1680
Now we are going to predict binders for  dataset HLA_DRB1_04_01 using alleles parameter ['HLA-DRA1*01:01-DRB1*04:01']
total processed peptides after filtering -  463
do same for decoy
Now we are going to predict binders for  dataset HLA_DRB1_04_01 using alleles parameter ['HLA-DRA1*01:01-DRB1*04:01']
total processed peptides after filtering -  1680
Now we are going to predict binders for  dataset HLA_DRB1_04_05 using alleles parameter ['HLA-DRA1*01:01-DRB1*04:05']
total processed peptides after filtering -  380
do same for decoy
Now we are going to predict binders for  dataset HLA_DRB1_04_05 using alleles parameter ['HLA-DRA1*01:01-DRB1*04:

In [65]:
ALLELES[0]

'HLA_DRB1_01_01'

In [66]:
import pandas as pd

### Join predictions for given allele

#### join by length

In [67]:
per_allele_total_binders_predictions = dict()
per_decoy_key_per_allele_total_non_binders_predictions = dict()
for allele_name, predictions in per_allele_binder_predictions.items():
    list_of_df = [prediction.to_dataframe() for prediction in predictions]
    df_binders_result = pd.concat(list_of_df, ignore_index=True)
    df_binders_result['binder'] = 1
    per_allele_total_binders_predictions[allele_name] = df_binders_result

for decoy_key in DECOY_PREFIXES:
    per_decoy_key_per_allele_total_non_binders_predictions[decoy_key] = dict()
    for allele_name, predictions in per_decoy_key_per_allele_non_binder_predictions[decoy_key].items():
        list_of_df = [prediction.to_dataframe() for prediction in predictions]
        df_non_binders_result = pd.concat(list_of_df, ignore_index=True)
        df_non_binders_result['binder'] = 0
        per_decoy_key_per_allele_total_non_binders_predictions[decoy_key][allele_name] = df_non_binders_result
    
    
    
    

#### join binders/non_binders

In [68]:
per_allele_binder_predictions.keys()

dict_keys(['HLA_DRB1_01_01', 'HLA_DRB1_04_01', 'HLA_DRB1_04_05'])

In [69]:
per_decoy_key_per_allele_total_predictions = dict()
for decoy_key in DECOY_PREFIXES:
    per_decoy_key_per_allele_total_predictions[decoy_key] = dict()
    for allele_name in per_allele_binder_predictions.keys():
        df1 = per_allele_total_binders_predictions[allele_name]
        df2 = per_decoy_key_per_allele_total_non_binders_predictions[decoy_key][allele_name]
        result_df =  pd.concat([df1, df2], ignore_index=True)
        per_decoy_key_per_allele_total_predictions[decoy_key][allele_name] = result_df

In [70]:
for decoy_key in DECOY_PREFIXES:
    for allele_name, total_df in per_decoy_key_per_allele_total_predictions[decoy_key].items():
        total_df.to_csv(f"/home/denklewer/data//mhc_predictions_{allele_name}_{decoy_key}")

In [71]:
per_decoy_key_per_allele_total_predictions['x1'][ALLELES[0]].describe()


,offset,score,percentile_rank,length,binder
count,3113.0,3113.000000,3113.000000,3113.000000,3113.000000
mean,0.0,0.146333,48.163836,16.740122,0.460328
std,0.0,0.301251,33.921951,3.456243,0.498504
min,0.0,0.000001,0.000000,9.000000,0.000000
25%,0.0,0.000841,14.920000,15.000000,0.000000
50%,0.0,0.003560,49.790000,15.000000,0.000000
75%,0.0,0.048115,80.140000,19.000000,1.000000
max,0.0,0.997930,95.000000,36.000000,1.000000
